In [ ]:
!pip install pygrog[dev]


# Gadgets with mri-nufft Data: Subspace and B0 ORC

This example uses a common data pipeline for both PyGROG gadgets:

1. BrainWeb phantom from ``brainweb-dl``.
2. Trajectory + k-space simulation + reference adjoint from ``mri-nufft``.
3. GROG gridding to feed :class:`~pygrog.operator.SparseFFT`.
4. Comparison against mri-nufft reference formulations.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from mrinufft import display_2D_trajectory, get_operator, initialize_2D_spiral
from mrinufft.density import voronoi
from mrinufft.extras import fse_simulation, get_brainweb_map, make_b0map
from mrinufft.extras.smaps import coil_compression
from mrinufft.operators.off_resonance import MRIFourierCorrected
from mrinufft.trajectories.utils import Acquisition
from pygrog.calib import GrogInterpolator
from pygrog.gadgets import OffResonanceCorrection, SubspaceProjection, SubspaceSparseFFT
from pygrog.operator import SparseFFT

## Shared setup



In [ ]:
def _synthetic_smaps(shape, n_coils=4):
    ny, nx = shape
    yy, xx = np.mgrid[-1 : 1 : ny * 1j, -1 : 1 : nx * 1j]
    smaps = []
    for angle in np.linspace(0.0, 2.0 * np.pi, n_coils, endpoint=False):
        cx, cy = np.cos(angle), np.sin(angle)
        gauss = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * 0.45**2))
        phase = np.exp(1j * (cx * xx + cy * yy))
        smaps.append(gauss * phase)
    smaps = np.asarray(smaps, dtype=np.complex64)
    smaps /= np.sqrt((np.abs(smaps) ** 2).sum(0, keepdims=True)) + 1e-12
    return smaps


m0, t1, t2 = get_brainweb_map(0)
m0 = np.flip(m0, axis=(0, 1, 2))[90].astype(np.float32)
t1 = np.flip(t1, axis=(0, 1, 2))[90].astype(np.float32)
t2 = np.flip(t2, axis=(0, 1, 2))[90].astype(np.float32)

m0 /= m0.max() + 1e-8
shape = m0.shape
n_coils_sim = 32  # coils for simulation
n_coils_cc = 8    # virtual coils after PCA compression

samples = initialize_2D_spiral(Nc=64, Ns=600, nb_revolutions=10, tilt="mri-golden").astype(
    np.float32
)
density = voronoi(samples)

display_2D_trajectory(samples)
plt.show()

# Simulate one multi-coil acquisition with ground-truth smaps.
smaps_true = _synthetic_smaps(shape, n_coils=n_coils_sim)
nufft_sim = get_operator("finufft")(
    samples=samples,
    shape=shape,
    n_coils=n_coils_sim,
    smaps=smaps_true,
    density=density,
    squeeze_dims=True,
)
kspace_single = nufft_sim.op(m0.astype(np.complex64))  # (n_coils_sim, n_samples)

# Coil compression 32 -> 8 virtual coils: better conditioning than raw 8 coils
kspace_single_cc, cc_matrix = coil_compression(
    kspace_single, K=n_coils_cc, traj=samples.reshape(-1, 2)
)
smaps_cc = (cc_matrix @ smaps_true.reshape(n_coils_sim, -1)).reshape(n_coils_cc, *shape)
smaps_cc = smaps_cc.astype(np.complex64)
smaps_cc /= np.sqrt((np.abs(smaps_cc) ** 2).sum(0, keepdims=True)) + 1e-12

# GROG plan and SparseFFT base operator for PyGROG gadgets.
coords = (samples * np.asarray(shape, dtype=np.float32)).astype(np.float32)
coil_calib = smaps_cc * m0[None, ...]
calib_cart_full = np.fft.fftshift(
    np.fft.fftn(np.fft.ifftshift(coil_calib, axes=(-2, -1)), axes=(-2, -1)),
    axes=(-2, -1),
).astype(np.complex64)
# Extract 24x24 calibration region from k-space centre
calib_size = 24
cy, cx = shape[0] // 2, shape[1] // 2
calib_cart = calib_cart_full[
    :,
    cy - calib_size // 2 : cy + calib_size // 2,
    cx - calib_size // 2 : cx + calib_size // 2,
]

grog = GrogInterpolator(shape=shape, coords=coords, kernel_width=2, oversamp=2.0, image_shape=shape)
grog.calc_interp_table(calib_cart, lamda=0.01, precision=1)

base_op = SparseFFT(
    plan=grog.plan, smaps=torch.as_tensor(smaps_cc)
)

## Gadget 1: SubspaceProjection / SubspaceSparseFFT



In [ ]:
etl = 8
te = np.arange(etl, dtype=np.float32) * 8.0
tr = 3000.0

frames = fse_simulation(m0, t1, t2, te, tr).astype(np.float32)
frames = np.ascontiguousarray(frames)  # (T, ny, nx)

# Simulate non-Cartesian k-space per frame using mri-nufft.
kspace_frames = np.stack(
    [nufft_sim.op(frames[t].astype(np.complex64)) for t in range(etl)],
    axis=0,
)  # (T, n_coils_sim, n_samples)
# Compress each frame to n_coils_cc virtual coils
kspace_frames = np.stack(
    [(cc_matrix @ kspace_frames[t]).astype(np.complex64) for t in range(etl)],
    axis=0,
)  # (T, n_coils_cc, n_samples)

# mri-nufft adjoint reference per frame.
nufft_ref = get_operator("finufft")(
    samples=samples,
    shape=shape,
    n_coils=n_coils_cc,
    smaps=smaps_cc,
    density=density,
    squeeze_dims=True,
)
frames_ref = np.stack([nufft_ref.adj_op(kspace_frames[t]) for t in range(etl)], axis=0)


# Learn subspace basis from signal dictionary sampled from BrainWeb ranges.
def _estimate_basis(train_data, rank):
    _, _, vh = np.linalg.svd(train_data, full_matrices=False)
    return vh[:rank]


t1_vals = np.linspace(float(t1[t1 > 0].min()) + 1.0, float(t1.max()), 60)
t2_vals = np.linspace(float(t2[t2 > 0].min()) + 1.0, float(t2.max()), 60)
t1_grid, t2_grid = np.meshgrid(t1_vals, t2_vals)
train = fse_simulation(1.0, t1_grid.ravel(), t2_grid.ravel(), te, tr).astype(np.float32)
rank = 4
basis = _estimate_basis(train.T, rank)

# mri-nufft subspace reference: manual adjoint (∑_t conj(φ_r(t)) * NUFFT^H y_t)
# This avoids depending on MRISubspace's internal batching convention.
coeff_ref_nufft = np.stack(
    [
        sum(
            np.conj(basis[r, t]) * nufft_ref.adj_op(kspace_frames[t])
            for t in range(etl)
        )
        for r in range(rank)
    ],
    axis=0,
)

# PyGROG subspace path: non-Cartesian -> sparse Cartesian -> SparseFFT + subspace.
# Pre-multiply by plan.pre_weights once here (caller's responsibility) so the
# orthonormality condition holds inside SubspaceSparseFFT.
sqrt_w = grog.plan.pre_weights
kspace_sparse = []
for t in range(etl):
    kspace_t = (
        kspace_frames[t].astype(np.complex64).reshape(n_coils_cc, *samples.shape[:2])
    )
    sparse_t = torch.as_tensor(np.asarray(grog.interpolate(kspace_t, ret_image=False)))
    kspace_sparse.append(sparse_t * sqrt_w.to(sparse_t.dtype).unsqueeze(0))
kspace_sparse = torch.stack(kspace_sparse, dim=0)  # (T, n_coils, n_samples)

proj = SubspaceProjection(n_components=rank)
proj.fit(torch.as_tensor(train, dtype=torch.float32))
sub_op = SubspaceSparseFFT(base_op, proj.basis.to(torch.complex64))
coeff_pygrog = sub_op.forward(kspace_sparse).detach().cpu().numpy()

# Display first coefficient comparison.
c_ref = np.abs(coeff_ref_nufft[0])
c_ref /= c_ref.max() + 1e-12
c_grog = np.abs(coeff_pygrog[0])
c_grog /= c_grog.max() + 1e-12

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(c_ref, cmap="gray", origin="lower")
axes[0].set_title("mri-nufft subspace coeff #1")
axes[0].axis("off")
axes[1].imshow(c_grog, cmap="gray", origin="lower")
axes[1].set_title("PyGROG subspace coeff #1")
axes[1].axis("off")
axes[2].imshow(c_grog - c_ref, cmap="bwr", origin="lower", vmin=-0.2, vmax=0.2)
axes[2].set_title("Difference")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## Gadget 2: OffResonanceCorrection



In [ ]:
brain_mask = m0 > 0.1 * m0.max()
b0_map, _ = make_b0map(shape, b0range=(-120, 120), mask=brain_mask)
readout_time = (
    np.arange(samples.shape[1], dtype=np.float32) * Acquisition.default.raster_time
)
readout_time = np.repeat(readout_time[None, :], samples.shape[0], axis=0)

orc_nufft = MRIFourierCorrected(
    nufft_ref,
    b0_map=b0_map,
    readout_time=readout_time,
    mask=brain_mask,
)

kspace_off = orc_nufft.op(m0.astype(np.complex64))
image_ref_orc = np.abs(orc_nufft.adj_op(kspace_off))

# PyGROG ORC: sparse interpolation, then apply ORC-corrected SparseFFT.
# OffResonanceCorrection.adjoint expects (n_coils_cc, n_samples) -> (n_coils_cc, *image)
# so we use a no-smaps operator and combine coils manually afterwards.
base_op_orc = SparseFFT(plan=grog.plan)  # no smaps
sparse_off = torch.as_tensor(
    np.asarray(
        grog.interpolate(
            kspace_off.astype(np.complex64).reshape(n_coils_cc, *samples.shape[:2]),
            ret_image=False,
        )
    ),
    dtype=torch.complex64,
)
# Pre-multiply by plan.pre_weights (caller's responsibility before ORC adjoint).
sparse_off = sparse_off * sqrt_w.to(sparse_off.dtype).unsqueeze(0)

# Derive per-sparse-sample readout times: each original non-Cartesian sample
# maps to up to kw Cartesian neighbours, so expand readout_time accordingly
# and filter to the valid (non-sentinel) neighbour entries that match
# what grog.interpolate() returns.
_kw = grog.plan.target_idx.shape[-1]
_valid_np = grog.plan.valid_mask.numpy()
readout_time_sparse = np.repeat(readout_time.ravel(), _kw)[_valid_np].astype(np.float32)

orc_pygrog = OffResonanceCorrection(
    base_op_orc,
    field_map=b0_map.astype(np.float32),
    readout_time=readout_time_sparse,
    mask=brain_mask,
    n_components=8,
    method="svd",
)
# Adjoint returns (n_coils_cc, *image_shape) — combine with smaps
smaps_t = torch.as_tensor(smaps_cc)
result_orc = orc_pygrog.adjoint(sparse_off)  # (n_coils_cc, *image_shape)
image_pygrog_orc = np.abs((result_orc * smaps_t.conj()).sum(0).detach().cpu().numpy())

image_ref_orc /= image_ref_orc.max() + 1e-12
image_pygrog_orc /= image_pygrog_orc.max() + 1e-12

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_ref_orc, cmap="gray", origin="lower")
axes[0].set_title("mri-nufft ORC adjoint")
axes[0].axis("off")
axes[1].imshow(image_pygrog_orc, cmap="gray", origin="lower")
axes[1].set_title("PyGROG ORC adjoint")
axes[1].axis("off")
axes[2].imshow(
    image_pygrog_orc - image_ref_orc,
    cmap="bwr",
    origin="lower",
    vmin=-0.2,
    vmax=0.2,
)
axes[2].set_title("Difference")
axes[2].axis("off")
plt.tight_layout()
plt.show()